# Provisional UiO-66 paired UMA experiment
Use Internet, two T4 GPUs and Kaggle Secret `HF_TOKEN`. Attach
`UIO66_Adsorption_bundle.zip` as a **private Dataset**. Execute cells in order.
This is a separate model-only design study; no ODAC25 labels are replaced.
Single-molecule sampled binding energies are not equilibrium DAC selectivity.
Use a fresh session/output directory; never overwrite prior evidence.

In [ ]:
import os, sys, json, hashlib, zipfile, subprocess, time, shutil
from pathlib import Path
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'fairchem-core==2.23.0', 'ase==3.26.0'], check=True)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=True)

In [ ]:
matches = list(Path('/kaggle/input').rglob('UIO66_Adsorption_bundle.zip'))
assert len(matches) == 1, 'Attach exactly one adsorption bundle'
UIO_ROOT = Path('/kaggle/working/uio66_code')
assert not UIO_ROOT.exists(), 'Use a fresh extraction directory'
UIO_ROOT.mkdir()
with zipfile.ZipFile(matches[0]) as archive:
    assert all((UIO_ROOT/name).resolve().is_relative_to(UIO_ROOT) for name in archive.namelist())
    archive.extractall(UIO_ROOT)
for name, digest in json.loads((UIO_ROOT/'bundle_manifest.json').read_text())['sha256'].items():
    assert hashlib.sha256((UIO_ROOT/name).read_bytes()).hexdigest() == digest, name
uio_env = {**os.environ, 'PYTHONPATH': str(UIO_ROOT), 'OMP_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1'}

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
uma_revision = 'f611b917d9c68566bbbeccbb0aa0f7cad1696cb2'
filename = 'checkpoints/uma-s-1p2p1.pt'
assert filename in HfApi().list_repo_files('facebook/UMA', revision=uma_revision, token=os.environ['HF_TOKEN'])
uma_checkpoint = hf_hub_download('facebook/UMA', filename, revision=uma_revision, token=os.environ['HF_TOKEN'])
bare_reports = sorted((UIO_ROOT/'uio66_bare').glob('gpu*/relaxation.json'))
all_ids = sorted(row['id'] for report in bare_reports for row in json.loads(report.read_text())['results'])
assert len(all_ids) == len(set(all_ids)) == 64
ads_base = [sys.executable, str(UIO_ROOT/'kaggle/run_uio66_adsorption.py'),
    '--bare-reports', *map(str, bare_reports), '--checkpoint', uma_checkpoint,
    '--checkpoint-revision', uma_revision]
print('Pinned revision:', uma_revision, 'bare configurations:', len(all_ids))

## Pilot
Inspect two configurations, two starts per gas. Both pilot groups were reserved
for training before adsorption labels were observed. Failures remain in results.
Pilot output must pass the independent trajectory audit before the full run.

In [ ]:
pilot = Path('/kaggle/working/uio66_ads_pilot')
assert not pilot.exists()
subprocess.run(ads_base + ['--output', str(pilot/'adsorption.json'), '--starts', '2',
    '--ids', 'uio66_000000', 'uio66_111111'], env={**uio_env, 'CUDA_VISIBLE_DEVICES': '0'}, check=True)
subprocess.run([sys.executable, '-m', 'scripts.audit_uio66_adsorption', '--report',
    str(pilot/'adsorption.json'), '--output', str(pilot/'audit.json')], cwd=UIO_ROOT, env=uio_env, check=True)
pilot_audit = json.loads((pilot/'audit.json').read_text())
print(pilot_audit['status_counts'])
assert pilot_audit['status_counts'] == {'complete_model_sample': 2}, 'Inspect failed diagnostics before proceeding'

## Main experiment
All 64 states, four starts per gas, identical 200-step / 0.05 eV/Angstrom limits.
Four independent processes, two per T4, single CPU thread each. Do not run the
cell twice, use Run All on a live run, or interrupt/reload a busy kernel.
Reports are atomic checkpoints. Process exit zero does not prove all targets passed.

In [ ]:
main = Path('/kaggle/working/uio66_ads_full')
assert not main.exists(), 'Preserve prior runs; choose a new experiment path'
ads_jobs = []
for worker in range(4):
    folder = main/f'worker{worker}'
    folder.mkdir(parents=True)
    logfile = (folder/'execution.log').open('w')
    command = ads_base + ['--output', str(folder/'adsorption.json'), '--starts', '4', '--ids', *all_ids[worker::4]]
    process = subprocess.Popen(command, env={**uio_env, 'CUDA_VISIBLE_DEVICES': str(worker//2)},
        stdout=logfile, stderr=subprocess.STDOUT)
    ads_jobs.append((process, logfile, folder))
while True:
    progress = []
    for process, _, folder in ads_jobs:
        report = folder/'adsorption.json'
        data = json.loads(report.read_text()) if report.exists() else {}
        progress.append({'worker': folder.name, 'finished': len(data.get('results', [])),
            'complete_pairs': sum(r['status']=='complete_model_sample' for r in data.get('results', [])),
            'active': (data.get('active_candidate') or {}).get('id'), 'return_code': process.poll()})
    print('Full adsorption progress:', progress, flush=True)
    if all(process.poll() is not None for process, _, _ in ads_jobs):
        break
    time.sleep(30)
ads_codes = [process.wait() for process, _, _ in ads_jobs]
for _, logfile, _ in ads_jobs:
    logfile.close()
print('Worker return codes:', ads_codes)
assert ads_codes == [0, 0, 0, 0], 'Preserve logs and inspect failures'

In [ ]:
reports = sorted(main.glob('worker*/adsorption.json'))
assert len(reports) == 4
for report in reports:
    subprocess.run([sys.executable, '-m', 'scripts.audit_uio66_adsorption', '--report', str(report),
        '--output', str(report.parent/'audit.json')], cwd=UIO_ROOT, env=uio_env, check=True)
ids = [r['id'] for report in reports for r in json.loads(report.read_text())['results']]
assert len(ids) == len(set(ids)) == 64 and set(ids) == set(all_ids)
for name in ['mof_dac/adsorption.py', 'mof_dac/relaxation.py', 'kaggle/run_uio66_adsorption.py',
             'scripts/audit_uio66_adsorption.py']:
    target = main/'executed_source'/name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(UIO_ROOT/name, target)
    for report in reports:
        expected = json.loads(report.read_text())['source_sha256'].get(name)
        if expected:
            assert hashlib.sha256(target.read_bytes()).hexdigest() == expected
archive = shutil.make_archive('/kaggle/working/uio66_ads_full_results', 'zip', main)
print('Download from Output:', archive)
print('Status counts:', {report.parent.name: json.loads((report.parent/'audit.json').read_text())['status_counts'] for report in reports})

Download `uio66_ads_full_results.zip` through Output → Refresh → file menu
→ Download. Keep the raw trajectories, reports, logs and executed source.
If Kaggle reports a draft save conflict, download the notebook backup; do not
reload while calculations run. No fit is valid until every frozen target passes.

In [ ]:
fit_output = Path('/kaggle/working/uio66_uma_fit')
subprocess.run([sys.executable, '-m', 'scripts.fit_uio66_adsorption', '--reports', *map(str, reports),
    '--output-dir', str(fit_output)], cwd=UIO_ROOT, env=uio_env, check=True)
print('Fit:', fit_output/'fit.json', 'provisional coefficient graph:', fit_output/'instance.json')
print('Review held-out errors and site-sampling ranges before solver/material claims.')